# 13. Цетановое число, W70/F30 и SHAP

Сравнение моделей с density proxy и без него, проверка SHAP-pruning и сценарной границы 51.

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px
from IPython.display import display
HERE=Path.cwd().resolve()
EDA=HERE if HERE.name=='eda' else HERE/'eda'
DIRECT=EDA/'experiments'/'cetane_density_shap_20260915'
RESID=EDA/'experiments'/'cetane_residual_density_shap_20260915'
metrics=pd.read_csv(RESID/'metrics.csv')
predictions=pd.read_csv(RESID/'predictions.csv')
shap=pd.read_csv(RESID/'shap_importance.csv')
intervals=pd.read_csv(RESID/'scenario_51_intervals.csv')

## Сравнение моделей на 2026

In [ ]:
evaluation=metrics[metrics.split=='evaluation'].groupby('variant').agg(mae_mean=('mae','mean'),mae_std=('mae','std'),rmse_mean=('rmse','mean')).reset_index().sort_values('mae_mean')
display(evaluation)
fig=px.bar(evaluation,x='variant',y='mae_mean',error_y='mae_std',title='MAE цетанового числа на шести пробах 2026')
fig.update_xaxes(tickangle=35); fig.show()

## Добавочная ценность W70/F30

In [ ]:
compare=metrics[(metrics.split=='evaluation')&metrics.variant.isin(['residual_process_no_density','residual_process_plus_density'])]
pivot=compare.pivot(index='seed',columns='variant',values='mae')
pivot['delta_mae_density_minus_no_density']=pivot['residual_process_plus_density']-pivot['residual_process_no_density']
display(pivot)
px.scatter(pivot.reset_index(),x='residual_process_no_density',y='residual_process_plus_density',hover_data=['seed'],title='Парное сравнение seed: с W70/F30 и без него').show()

## Устойчивость SHAP

In [ ]:
stability=shap.groupby(['feature','is_density_proxy']).agg(mean_rank=('rank','mean'),median_rank=('rank','median'),top10_share=('rank',lambda x:(x<=10).mean()),mean_abs_shap=('mean_abs_shap','mean')).reset_index().sort_values(['top10_share','mean_rank'],ascending=[False,True])
display(stability.head(30))
display(stability[stability.is_density_proxy].sort_values('mean_rank'))
px.bar(stability.head(20),x='mean_abs_shap',y='feature',orientation='h',color='top10_share',title='Устойчивые SHAP-признаки по 10 seed').show()

## Сценарная проверка нижней границы 51

Точечные прогнозы не обнаружили единственное нарушение. Нижняя граница 90% интервала полной residual-модели обнаружила его, но в среднем предупреждала на четырёх из шести проб.

In [ ]:
display(intervals.groupby('variant')[['halfwidth90','coverage90','alarms','tp','fp','fn']].mean().sort_values('coverage90',ascending=False))
seed42=predictions[(predictions.seed==42)&(predictions.variant=='residual_process_plus_density')&predictions.split.eq('evaluation')].copy()
fig=px.line(seed42,x='timestamp',y=['actual','previous_lims','prediction'],markers=True,title='Цетановое число: фактическое, последнее ЛИМС и прогноз')
fig.add_hline(y=51,line_dash='dash',line_color='red'); fig.show()